In [4]:
import pandas as pd

# PERSIAPAN: Membaca Data
dim_stasiun = pd.read_csv('dim_stasiun.csv')
fact_ispu = pd.read_csv('fact_ispu.csv')

# TAHAP 1: Data Preparation (Wrangling & Cleansing)
# 1. Menggabungkan tabel (JOIN) berdasarkan id_stasiun
df_merged = pd.merge(fact_ispu, dim_stasiun, on='stasiun_id', how='inner')
# 2. Mengubah tipe data 'tanggal' menjadi tipe datetime agar mudah difilter
df_merged['tanggal'] = pd.to_datetime(df_merged['tanggal'])
# 3. Cleansing: Filter khusus tahun 2021 dan membuang nilai NULL pada 'pm25'
df_clean = df_merged[
    (df_merged['tanggal'].dt.year == 2021) &
    (df_merged['pm25'].notna())
].copy()
# 4. Memilih kolom yang dibutuhkan
kolom_pilihan = ['nama_stasiun', 'tanggal', 'categori', 'pm25'] 
df_clean = df_clean[kolom_pilihan]


# TAHAP 2: Analisis Tren 
# Mengurutkan data berdasarkan stasiun dan tanggal sebelum menghitung
df_clean = df_clean.sort_values(by=['nama_stasiun', 'tanggal']).reset_index(drop=True)
# Menghitung Moving Average 7 hari (menggunakan rolling)
# Min_periods=1 agar hari pertama-keenam tetap dihitung rata-ratanya walau belum genap 7 hari
df_clean['moving_avg_pm25_7hari'] = (
    df_clean.groupby('nama_stasiun')['pm25']
    .transform(lambda x: x.rolling(window=7, min_periods=1).mean())
)

print("Hasil Tahap 1 & 2: ")
display(df_clean.head())

# TAHAP 3: Statistik Dasar 
# Menghitung rata-rata, nilai tertinggi, dan terendah per stasiun
statistik_dasar = df_clean.groupby('nama_stasiun').agg(
    rata_rata_tahunan_pm25=('pm25', 'mean'),
    pm25_tertinggi=('pm25', 'max'),
    pm25_terendah=('pm25', 'min')
).reset_index()
# Membulatkan angka desimal ke 2 angka di belakang koma dan mengurutkan
statistik_dasar['rata_rata_tahunan_pm25'] = statistik_dasar['rata_rata_tahunan_pm25'].round(2)
statistik_dasar = statistik_dasar.sort_values(by='rata_rata_tahunan_pm25', ascending=False).reset_index(drop=True)
print("\n Hasil Tahap 3: Statistik Dasar")
display(statistik_dasar)

Hasil Tahap 1 & 2: 


,nama_stasiun,tanggal,categori,pm25,moving_avg_pm25_7hari
0,DKI1 (Bunderan HI),2021-12-21,SEDANG,74.0,74.000000
1,DKI2 (Kelapa Gading),2021-01-01,SEDANG,58.0,58.000000
2,DKI2 (Kelapa Gading),2021-01-04,SEDANG,49.0,53.500000
3,DKI2 (Kelapa Gading),2021-01-06,SEDANG,60.0,55.666667
4,DKI2 (Kelapa Gading),2021-01-07,SEDANG,50.0,54.250000



 Hasil Tahap 3: Statistik Dasar


,nama_stasiun,rata_rata_tahunan_pm25,pm25_tertinggi,pm25_terendah
0,DKI4 (Lubang Buaya),100.19,174.0,45.0
1,DKI5 (Kebon Jeruk),92.61,140.0,52.0
2,DKI3 (Jagakarsa),82.73,126.0,51.0
3,DKI1 (Bunderan HI),74.00,74.0,74.0
4,DKI2 (Kelapa Gading),67.86,119.0,29.0
